# HIV bnAb figures: individual panels

Builds each panel of the composite as its own figure for **1NC9** and **CH235.12**.

- **A** antigen vs antigen scatter, coloured by the S56R + T57P switch state
- **B** boxplots by mature-residue count, split by S56R
- **C** the same scatter faceted into the four switch states, coloured by background
- **D** half-violins split by S56R, grouped on T57P
- **E** inferred affinity landscape over the panel sub-cube


## Imports

In [ ]:
import os, itertools, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from matplotlib.gridspec import GridSpec
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.patches import Patch, Rectangle
from matplotlib.lines import Line2D

plt.rcParams.update({"font.family":"DejaVu Sans","font.size":9,"axes.linewidth":0.7,
    "mathtext.default":"regular","xtick.major.size":3,"ytick.major.size":3,
    "xtick.major.width":0.7,"ytick.major.width":0.7,
    "svg.fonttype":"path", "pdf.fonttype":42})

## Configuration

In [ ]:
data_dir  = "../data"
fig_dir   = "../figures"
os.makedirs(fig_dir, exist_ok=True)
sites     = [f"site_{i}" for i in range(10)]
antigens  = ["CH505TF", "SF162"]
name_to_stem = {"1NC9":"1NC9", "CH235.12":"CH235_12"}   # display name -> file stem
png_dpi   = 300
scatter_seed = 0

floor_kd = 6.0               # assay detection floor
axis_lo, axis_hi = 5.6, 8.25 # scatter axis span
clu_kd = round((axis_lo + floor_kd) / 2, 3)   # censored points sit here, one gap-separated band

antigen_hex = {"CH505TF":"#E66100", "SF162":"#5D3A9B"}

# background key colours, indexed by subset of the three background sites in
# size-then-panel-order: (), (b0,), (b1,), (b2,), (b0,b1), (b0,b2), (b1,b2), (b0,b1,b2)
bg_hex = ["#5c5e9b","#27489c","#72afe3","#397057","#caa94a","#ad5b46","#c77ea3","#6c1b48"]

# panel A switch-state colours, drawn from the same key
switch_hex = {"neither":"#72afe3", "S56R only":"#caa94a",
              "T57P only":"#397057", "both":"#6c1b48"}

# violin fills as (switch absent, switch present) pairs, same subset indexing as bg_hex
violin_hex = [("#8dbfe0","#2886c5"), ("#cce0f3","#9fc6e9"), ("#f7b890","#f0792d"),
              ("#bdb1da","#836cba"), ("#9dd29d","#46ab46"), ("#c0a8a5","#885b56"),
              ("#e2939c","#c93244"), ("#f7c1c6","#f18a93")]

bubble_cmap = "mako_r"   # low value -> light (none/germline), high -> dark (mature)
bubble_y_step = 3.3      # vertical spacing between cube levels; larger spreads nodes out
bg_mut = "S56R"          # site the violins are split on, left half absent, right half present

# three background sites keying panels C and D, and the site the violins group on
facet_bg  = {"1NC9":["N52D","S53R","G54R"], "CH235.12":["M34I","N52D","E81D"]}
group_mut = {"1NC9":"T57P", "CH235.12":"T57P"}
bubble_panel = {"1NC9":["N52D","S53R","G54R","S56R","T57P"],
                "CH235.12":["M34I","N52D","T57P","E81D"]}
bubble_split = {"1NC9":None, "CH235.12":"S56R"}

lab_meas = r"Measured Binding Affinity ($-\log_{10}K_D$)"
lab_inf  = r"Inferred Binding Affinity ($-\log_{10}\hat{K}_D$)"

## Data helpers

In [ ]:
def meta_maps(ab):
    """site labels (e.g. S56R), regions, and label -> site index."""
    m = pd.read_csv(f"{data_dir}/{name_to_stem[ab]}_meta.csv")
    m = m.assign(_i=m["site"].str.replace("site_","",regex=False).astype(int)).sort_values("_i")
    labels  = [f"{r.germline_aa}{r.kabat_position}{r.mature_aa}" for r in m.itertuples()]
    regions = [r.region for r in m.itertuples()]
    return labels, regions, {l:i for i,l in enumerate(labels)}

def kabat_labels(ab):
    """site_i -> kabat label, for genotype keys and dot-grid rows."""
    m = pd.read_csv(f"{data_dir}/{name_to_stem[ab]}_meta.csv")
    return {s: f"{m.loc[r,'germline_aa']}{m.loc[r,'kabat_position']}{m.loc[r,'mature_aa']}"
            for r, s in enumerate(m["site"].values)}

def df_all(ab, ag):
    # full inferred-landscape table for one antibody + antigen
    return pd.read_csv(f"{data_dir}/{name_to_stem[ab]}_{ag}_BEST_df_all.csv").drop_duplicates("genotype_tokens")

def token_kds(ab, ag):
    # measured nlog10_Kd per genotype token for one antibody + antigen
    k = pd.read_csv(f"{data_dir}/{name_to_stem[ab]}_kds.csv"); k = k[k.antigen == ag]
    return k[sites + ["nlog10_Kd"]].copy()

def variant_xy(ab):
    """one row per variant: replicate-averaged affinity against both antigens, plus its token."""
    k = pd.read_csv(f"{data_dir}/{name_to_stem[ab]}_kds.csv")
    k["tok"] = k[sites].agg("".join, axis=1)
    def avg(ag):
        d = k[k.antigen == ag]
        spec = {"v":("nlog10_Kd","mean"), "tok":("tok","first")}
        for s in sites: spec[s] = (s, "first")
        return d.groupby("id", as_index=False).agg(**spec)
    ch = avg("CH505TF").rename(columns={"v":"ch"}); sf = avg("SF162").rename(columns={"v":"sf"})
    return ch.merge(sf[["id","sf"]], on="id")

def bg_subsets(ab):
    """the eight background subsets, size then panel order, as tuples of labels."""
    b = facet_bg[ab]
    return [tuple(c) for r in range(len(b)+1) for c in itertools.combinations(b, r)]

def bg_index(ab, tok):
    """which background subset a genotype token belongs to."""
    n2s = meta_maps(ab)[2]
    on = tuple(x for x in facet_bg[ab] if tok[n2s[x]] == "M")
    return bg_subsets(ab).index(on)

## Shared drawing helpers

In [ ]:
def cluster_floor(v, rng):
    # move censored values into a gap-separated band below the floor
    v = np.asarray(v, float).copy(); f = np.isclose(v, floor_kd)
    v[f] = clu_kd + rng.uniform(-.055, .055, f.sum())
    return v

# Curly brace just outside an axis, in data coords along the axis and axes coords across it
def brace(ax, span, at, which, depth=0.035, label=None, lw=0.9, pad=0.012):
    lo, hi = span
    n = 101; half = n // 2 + 1
    t = np.linspace(lo, hi, n); th = t[:half]
    beta = 44.0 / max(hi - lo, 1e-9)
    b = (1/(1+np.exp(-beta*(th-th[0]))) + 1/(1+np.exp(-beta*(th-th[-1]))))
    b = np.concatenate((b, b[-2::-1])); b = b - b.min(); b = b / max(b.max(), 1e-9)
    off = at - depth*b
    if which == "x":
        ax.plot(t, off, transform=ax.get_xaxis_transform(), color="black", lw=lw,
                clip_on=False, zorder=6, solid_capstyle="round")
        if label: ax.text((lo+hi)/2, at-depth-pad, label, transform=ax.get_xaxis_transform(),
                          ha="center", va="top", fontsize=8, clip_on=False)
    else:
        ax.plot(off, t, transform=ax.get_yaxis_transform(), color="black", lw=lw,
                clip_on=False, zorder=6, solid_capstyle="round")
        if label: ax.text(at-depth-pad, (lo+hi)/2, label, transform=ax.get_yaxis_transform(),
                          ha="right", va="center", fontsize=8, clip_on=False)

def group_brace(ax, x0, x1, label, y=1.03, depth=0.035):
    """curly brace above the axes marking a column group, with a bold label."""
    n = 201; half = n // 2 + 1
    t = np.linspace(x0, x1, n); th = t[:half]
    beta = 90.0 / max(x1 - x0, 1e-9)
    b = (1/(1+np.exp(-beta*(th-th[0]))) + 1/(1+np.exp(-beta*(th-th[-1]))))
    b = np.concatenate((b, b[-2::-1])); b = b - b.min(); b = b / max(b.max(), 1e-9)
    ax.plot(t, y + depth*b, transform=ax.get_xaxis_transform(), color="black", lw=1.1,
            clip_on=False, zorder=6, solid_capstyle="round")
    ax.text((x0+x1)/2, y+depth+0.022, label, transform=ax.get_xaxis_transform(),
            ha="center", va="bottom", fontsize=10.5, fontweight="bold", clip_on=False)

# Color strip over a genotype dot grid with an n row, used by panels A and C
def draw_key(ax, colours, rows, row_states, counts, markers=None):
    ncol = len(colours); nrow = len(rows)
    for i, (mark, size, measured, label) in enumerate(markers or []):
        x = ncol * (0.16 + 0.42*i)
        fill = dict(color="black") if measured else dict(facecolors="white", edgecolors="black", linewidths=1.1)
        ax.scatter([x],[nrow+1.55], s=size, marker=mark, clip_on=False, zorder=4, **fill)
        ax.text(x+0.13, nrow+1.55, label, ha="left", va="center", fontsize=8.5)
    for i, c in enumerate(colours):
        ax.add_patch(Rectangle((i, nrow+0.35), 1.0, 0.55, facecolor=c, edgecolor="none", zorder=3))
    for r, name in enumerate(rows):
        yy = nrow - 1 - r
        ax.text(-0.18, yy, name, ha="right", va="center", fontsize=8)
        for i in range(ncol):
            on = row_states[i][r]
            ax.scatter(i+0.5, yy, s=26, color="#222222" if on else "#e2e2e2", zorder=3)
    ax.text(-0.18, -1.05, "n=", ha="right", va="center", fontsize=8)
    for i, n in enumerate(counts):
        ax.text(i+0.5, -1.05, str(n), ha="center", va="center", fontsize=8)
    ax.set_xlim(-0.05, ncol+0.05); ax.set_ylim(-1.7, nrow + (2.2 if markers else 1.1))
    ax.set_xticks([]); ax.set_yticks([]); ax.axis("off")

# Diagonal, floor guides, censored ticks and braces used across scatters
def scatter_frame(ax, xlabel=True, ylabel=True, xticks=True, yticks=True):
    ax.plot([axis_lo,axis_hi],[axis_lo,axis_hi], ls="--", color="#B0B0B0", lw=.9, zorder=0)
    ax.axhline(floor_kd, ls=":", color="#d1a0a0", lw=.8, zorder=0)
    ax.axvline(floor_kd, ls=":", color="#d1a0a0", lw=.8, zorder=0)
    ax.set_xlim(axis_lo,axis_hi); ax.set_ylim(axis_lo,axis_hi); ax.set_aspect("equal")
    band = (clu_kd-0.14, clu_kd+0.14)
    ax.set_xticks([6.5,7,7.5,8])
    ax.set_xticklabels(["6.5","7.0","7.5","8.0"] if xticks else [])
    ax.set_yticks([6.5,7,7.5,8])
    ax.set_yticklabels(["6.5","7.0","7.5","8.0"] if yticks else [])
    if xticks: brace(ax, band, -0.012, "x", depth=0.030, label=r"$\leq 6.0$")
    if yticks: brace(ax, band, -0.012, "y", depth=0.030, label=r"$\leq 6.0$")
    if xlabel: ax.set_xlabel("SF162\n"+lab_meas, fontsize=8, labelpad=16)
    if ylabel: ax.set_ylabel("CH505TF\n"+lab_meas, fontsize=8)

## Panel A: antigen vs antigen scatter
Replicate-averaged per variant, coloured by the S56R and T57P state. Censored values sit in a gap-separated band below the floor, braced on both axes.

In [ ]:
def switch_cat(tok, i56, i57):
    a, b = tok[i56] == "M", tok[i57] == "M"
    return "both" if a and b else "S56R only" if a else "T57P only" if b else "neither"

# Per-variant CH505TF vs SF162 affinity, coloured by the S56R / T57P state
def draw_scatter(ax, ab):
    n2s = meta_maps(ab)[2]; i56, i57 = n2s["S56R"], n2s["T57P"]
    m = variant_xy(ab); m["cat"] = m["tok"].apply(lambda t: switch_cat(t, i56, i57))
    rng = np.random.default_rng(scatter_seed)
    scatter_frame(ax)
    for c in ["neither","S56R only","T57P only","both"]:
        d = m[m.cat == c]
        ax.scatter(cluster_floor(d["sf"], rng), cluster_floor(d["ch"], rng), s=16,
                   color=switch_hex[c], alpha=.75, edgecolors="none", linewidths=0, zorder=2)
    draw_reference_markers(ab, ax, m, rng)
    return m

# reference points and the marker each one gets, filled when measured, hollow when inferred
ref_points = [("G"*10, "o", 60, 45, "germline"), ("M"*10, "*", 130, 110, "mature")]

def inferred_xy(ab, tok):
    """(SF162, CH505TF) predicted affinity for one genotype, for reference points with no measurement."""
    out = []
    for ag in ["SF162", "CH505TF"]:
        d = df_all(ab, ag)
        out.append(float(d.loc[d["genotype_tokens"].astype(str) == tok, "predicted_kd_full"].iloc[0]))
    return out

def reference_status(ab, m):
    """(marker, key size, measured, key label) per reference point, for the panel key."""
    return [(mark, ksize, bool(len(m[m["tok"] == tok])),
             name if len(m[m["tok"] == tok]) else f"{name} (inferred)")
            for tok, mark, _, ksize, name in ref_points]

# Germline dot and mature star, measured when both antigens recorded the variant, otherwise inferred
def draw_reference_markers(ab, ax, m, rng):
    for tok, mark, size, _, _ in ref_points:
        d = m[m["tok"] == tok]
        if len(d):
            ax.scatter(cluster_floor(d["sf"], rng), cluster_floor(d["ch"], rng), s=size,
                       marker=mark, c="black", edgecolors="white", linewidths=.7, zorder=5)
        else:
            x, y = inferred_xy(ab, tok)
            ax.scatter(cluster_floor([x], rng), cluster_floor([y], rng), s=size,
                       marker=mark, facecolors="white", edgecolors="black",
                       linewidths=1.1, zorder=5)

## Panel B: affinity by mature-residue count, split by S56R
The count is over the other nine sites, so both halves span 0 to 9 and line up for a direct comparison.

In [ ]:
# Top of the affinity range across antigens, for a shared y-limit
def b_ymax(ab):
    k = pd.read_csv(f"{data_dir}/{name_to_stem[ab]}_kds.csv"); k["tok"] = k[sites].agg("".join, axis=1)
    top = floor_kd
    for ag in antigens:
        avg = k[k.antigen == ag].groupby("tok")["nlog10_Kd"].mean()
        if len(avg): top = max(top, float(avg.max()))
    return top

# Boxplots with S56R fixed, counting the other nine sites so both panels span 0 to 9
def draw_boxplot_split(ax, ab, s56_state, ylim, ylabel=True, legend=True, xlabel=True):
    n2s = meta_maps(ab)[2]; i56 = n2s["S56R"]
    k = pd.read_csv(f"{data_dir}/{name_to_stem[ab]}_kds.csv"); k["tok"] = k[sites].agg("".join, axis=1)
    k = k[k["tok"].str[i56] == s56_state]                       # hold S56R identity fixed
    counts = list(range(0,10)); off = {"CH505TF":-0.20,"SF162":0.20}; w = 0.34
    other9 = lambda t: sum(1 for j, ch in enumerate(t) if j != i56 and ch == "M")   # exclude S56R
    for ag in antigens:
        avg = k[k.antigen == ag].groupby("tok")["nlog10_Kd"].mean()
        nm = avg.index.to_series().apply(other9)
        for c in counts:
            vals = avg[nm == c].to_numpy(float)
            if len(vals) == 0: continue
            p = c + off[ag]
            if len(vals) == 1:                     # single variant -> dot + mean line, not a degenerate box
                ax.plot([p-w/2,p+w/2],[vals[0]]*2, color="black", lw=1.0, zorder=4)
                ax.scatter([p],[vals[0]], s=22, color=antigen_hex[ag], edgecolors="black", linewidths=.5, zorder=5)
            else:
                bp = ax.boxplot([vals], positions=[p], widths=w, patch_artist=True, showfliers=True,
                                medianprops=dict(color="black",lw=1.0), whiskerprops=dict(lw=.7), capprops=dict(lw=.7))
                for patch in bp["boxes"]: patch.set_facecolor(antigen_hex[ag]); patch.set_edgecolor("black"); patch.set_linewidth(.5); patch.set_alpha(.9)
                for fli in bp["fliers"]: fli.set(marker="o", ms=4.0, mfc=antigen_hex[ag], mec="black", mew=.4, alpha=.85)
    ax.axhline(floor_kd, ls=":", color="#d1a0a0", lw=.9)
    ax.set_xticks(counts); ax.set_xticklabels(counts); ax.set_xlim(-0.6,9.6); ax.set_ylim(*ylim)
    if xlabel: ax.set_xlabel("number of other mature residues")
    if ylabel: ax.set_ylabel(lab_meas, fontsize=8)
    if legend:
        ax.legend(handles=[Patch(facecolor=antigen_hex[a], edgecolor="black", alpha=.9, label=a) for a in antigens],
                  frameon=False, fontsize=8.5, title="antigen", loc="upper left")

## Panel C: scatter faceted by the switch
The same variants as panel A, split into the four switch states and keyed to the three-site background rather than the switch.

In [ ]:
# One switch-state facet, points keyed to the background subset
def draw_facet_scatter(ax, ab, m, s56_state, s57_state, xedge, yedge):
    n2s = meta_maps(ab)[2]; i56, i57 = n2s["S56R"], n2s["T57P"]
    rng = np.random.default_rng(scatter_seed)
    scatter_frame(ax, xlabel=xedge, ylabel=yedge, xticks=xedge, yticks=yedge)
    d = m[(m["tok"].str[i56] == s56_state) & (m["tok"].str[i57] == s57_state)]
    for i in range(len(bg_subsets(ab))):
        s = d[d["bg"] == i]
        if not len(s): continue
        ax.scatter(cluster_floor(s["sf"], rng), cluster_floor(s["ch"], rng), s=13,
                   color=bg_hex[i], alpha=.8, edgecolors="none", linewidths=0, zorder=2)
    draw_reference_markers(ab, ax, m, rng)
    sign = lambda st: "+" if st == "M" else "\u2212"
    ax.set_title(f"{sign(s56_state)}S56R {sign(s57_state)}T57P", fontsize=9.5)

## Panel D: half-violins split by S56R
Left half S56R absent, right half present; columns are grouped into the two endpoints, then the six intermediate backgrounds without T57P, then the same six with it, so the two T57P groups read against each other column by column.

In [ ]:
# (Background subset index, group site state) per column, plus group spans and labels
def violin_columns(ab):
    subs = bg_subsets(ab); last = len(subs)-1
    mid = list(range(1, last))
    g1 = [(0,"G"), (0,"M"), (last,"G"), (last,"M")]
    g2 = [(i,"G") for i in mid]
    g3 = [(i,"M") for i in mid]
    gs = group_mut[ab]
    return [g1, g2, g3], ["none / all", f"\u2212{gs}", f"+{gs}"]

# Raw nlog10_Kd arrays per (background subset, group state, split state)
def violin_values(ab, ag):
    n2s = meta_maps(ab)[2]; i56 = n2s[bg_mut]; ig = n2s[group_mut[ab]]
    d = token_kds(ab, ag); d["tok"] = d[sites].agg("".join, axis=1)
    d["bg"] = d["tok"].apply(lambda t: bg_index(ab, t))
    out = {}
    for (bi, gstate, sstate), sub in d.groupby([d["bg"], d["tok"].str[ig], d["tok"].str[i56]]):
        out[(bi, gstate, sstate)] = sub["nlog10_Kd"].to_numpy(float)
    return out

def half_violin(ax, data, pos, side, color, width=0.44):
    # draw one side of a split violin, or a dot when too few points
    data = np.asarray(data, float); data = data[np.isfinite(data)]
    if len(data) == 0: return
    cx = pos - width*0.5 if side == "left" else pos + width*0.5
    if len(data) < 3:
        ax.scatter([cx],[np.mean(data)], s=15, facecolor="white", edgecolor="black", linewidths=.7, zorder=7); return
    v = ax.violinplot([data], positions=[pos], widths=width*2, showextrema=False)
    for b in v["bodies"]:
        vv = b.get_paths()[0].vertices
        vv[:,0] = np.clip(vv[:,0], pos-width, pos) if side == "left" else np.clip(vv[:,0], pos, pos+width)
        b.set_facecolor(color); b.set_edgecolor("black"); b.set_linewidth(.7); b.set_alpha(1.0)
    med, mn = np.median(data), np.mean(data)
    seg = (pos-width*.9, pos) if side == "left" else (pos, pos+width*.9)
    ax.plot(seg,[med,med], color="#555", lw=1.1, zorder=6)
    ax.scatter([cx],[mn], s=15, facecolor="white", edgecolor="black", linewidths=.7, zorder=7)

# S56R-absent vs S56R-present half-violins, grouped by the T57P state
def draw_violin(ax, ax_dot, ab, ag, legend=False, brackets=False, vy=(5.9,8.5), gap=1.1):
    kl = kabat_labels(ab); subs = bg_subsets(ab)
    groups, glabels = violin_columns(ab); vals = violin_values(ab, ag)
    xs = []; x = 0.0; spans = []
    for gi, grp in enumerate(groups):
        start = x
        for _ in grp: xs.append(x); x += 1.0
        spans.append((start, x-1.0)); x += gap
    flat = [c for grp in groups for c in grp]
    for xi, (bi, gstate) in zip(xs, flat):
        light, dark = violin_hex[bi % len(violin_hex)]
        half_violin(ax, vals.get((bi,gstate,"G"), []), xi, "left",  light)
        half_violin(ax, vals.get((bi,gstate,"M"), []), xi, "right", dark)
    for a, b in zip(spans[:-1], spans[1:]):
        xv = (a[1] + b[0]) / 2
        ax.axvline(xv, color="#555555", ls="--", lw=1.4, zorder=0)
    if brackets:
        for (s0, s1), lab in zip(spans, glabels):
            group_brace(ax, s0-0.5, s1+0.5, lab)
    ax.axhline(floor_kd, lw=.8, ls="--", color="#bbb", zorder=0)
    ax.set_ylim(*vy); ax.set_xlim(-0.7, xs[-1]+0.7); ax.set_xticks([])
    ax.set_yticks([6,6.5,7,7.5,8]); ax.set_yticklabels([r"$\leq 6$","6.5","7.0","7.5","8.0"])
    ax.set_ylabel(f"{ag}\n"+lab_meas, fontsize=7.5)
    if legend:
        ax.legend(handles=[Patch(facecolor="#d5d5d5", ec="k", lw=.6, label=f"\u2212{bg_mut}"),
                           Patch(facecolor="#8a8a8a", ec="k", lw=.6, label=f"+{bg_mut}"),
                           Line2D([0],[0], color="#555", lw=1.4, label="median"),
                           Line2D([0],[0], marker="o", ls="", mfc="white", mec="black", mew=.7, ms=7, label="mean")],
                  frameon=False, fontsize=7.5, loc="upper right", ncol=2,
                  handletextpad=.5, columnspacing=1.0)
    rows = facet_bg[ab] + [group_mut[ab]]; nrow = len(rows)
    for r, name in enumerate(rows):
        yy = nrow-1-r
        for xi, (bi, gstate) in zip(xs, flat):
            on = (name in subs[bi]) if r < len(facet_bg[ab]) else (gstate == "M")
            ax_dot.scatter(xi, yy, s=13, color="#222222" if on else "#e2e2e2", zorder=2)
    for xi, (bi, gstate) in zip(xs, flat):
        n = sum(len(vals.get((bi,gstate,s), [])) for s in ["G","M"])
        ax_dot.text(xi, -1.0, str(n), ha="center", va="top", fontsize=6, color="#262626")
    ax_dot.text(-0.85, -1.0, "n=", ha="right", va="top", fontsize=6, color="#262626")
    ax_dot.set_xlim(-0.7, xs[-1]+0.7); ax_dot.set_ylim(-1.9, nrow-0.4)
    ax_dot.set_xticks([]); ax_dot.set_yticks(range(nrow))
    ax_dot.set_yticklabels(rows[::-1], fontsize=7)
    for sp in ["top","right","bottom","left"]: ax_dot.spines[sp].set_visible(False)

## Panel E: inferred affinity landscape
Nodes over the panel sub-cube, `predicted_kd_uncensored`. Dark edge uphill, grey downhill. 1NC9 is unsplit; CH235.12 splits on S56R and averages over background.

In [ ]:
def subset_levels(k): return {l:[frozenset(c) for c in itertools.combinations(range(k),l)] for l in range(k+1)}

# (x, y) for each subset node, rows stacked by mutation level
def cube_layout(k, x_step, y_step):
    pos = {}
    for lvl, gs in subset_levels(k).items():
        m = len(gs); xs = [(i-(m-1)/2)*x_step for i in range(m)]
        for g, x in zip(gs, xs): pos[g] = (x, -lvl*y_step)
    return pos

# Background-averaged predicted_kd_uncensored with the focal site fixed
def avg_val_fn(ab, ag, sub_idx, focal_idx, focal_state):
    df = df_all(ab, ag)
    def f(present):
        d = df[df[f"site_{focal_idx}"] == focal_state]
        for li in range(len(sub_idx)):
            d = d[d[f"site_{sub_idx[li]}"] == ("M" if li in present else "G")]
        return float(np.mean(d["predicted_kd_uncensored"]))
    return f

# vmin at the none node, vmax at the all node, across antigens
def bubble_range(ab, subcube):
    n2s = meta_maps(ab)[2]; si = [n2s[s] for s in subcube]
    none_vals, all_vals = [], []
    for ag in antigens:
        df = df_all(ab, ag)
        d = df.copy()
        for idx in si: d = d[d[f"site_{idx}"] == "G"]
        none_vals.append(float(d["predicted_kd_uncensored"].mean()))
        d = df.copy()
        for idx in si: d = d[d[f"site_{idx}"] == "M"]
        all_vals.append(float(d["predicted_kd_uncensored"].mean()))
    return float(min(none_vals)), float(max(all_vals))

# Same none/all anchoring for split cubes, across antigens and both focal states
def bubble_range_split(ab, subcube, split_name):
    n2s = meta_maps(ab)[2]; si = [n2s[s] for s in subcube]; focal = n2s[split_name]
    none_vals, all_vals = [], []
    for ag in antigens:
        for st in ["G","M"]:
            fn = avg_val_fn(ab, ag, si, focal, st)
            none_vals.append(fn(())); all_vals.append(fn(tuple(range(len(si)))))
    return float(min(none_vals)), float(max(all_vals))

# One affinity-landscape cube, nodes coloured by inferred KD
def draw_bubble(ax, ab, ag, sub_idx, sub_lab, vmin, vmax, val_fn=None, title=None, y_step=None):
    if val_fn is None:
        lk = dict(zip(df_all(ab,ag)["genotype_tokens"].astype(str), df_all(ab,ag)["predicted_kd_uncensored"]))
        def val(present):
            ch = ["G"]*10
            for li in present: ch[sub_idx[li]] = "M"
            return float(lk["".join(ch)])
    else:
        val = val_fn
    k = len(sub_idx)
    x_step = 2.6 if k >= 5 else 3.0   # tighter spacing for the wide 5-site cube
    pos = cube_layout(k, x_step, y_step or bubble_y_step); allg = list(pos); gset = set(allg)
    norm = Normalize(vmin, vmax); cm = plt.get_cmap(bubble_cmap)
    for g in allg:
        for i in range(k):
            if i not in g and (g|{i}) in gset:
                downhill = val(tuple(sorted(g|{i}))) < val(tuple(sorted(g)))
                (x1,y1),(x2,y2) = pos[g], pos[g|{i}]
                ax.plot([x1,x2],[y1,y2], color=("#cfcfcf" if downhill else "#333333"),
                        lw=(.8 if downhill else 1.0), zorder=1)
    for g in allg:
        x, y = pos[g]; v = val(tuple(sorted(g))); col = cm(norm(v))
        ax.add_patch(plt.Circle((x,y), 1.2, facecolor=col, edgecolor="black", lw=1.1, zorder=2))
        lum = .299*col[0] + .587*col[1] + .114*col[2]; tc = "black" if lum > .6 else "white"
        present = sorted(g)
        txt = "none" if not present else ("all" if len(present)==k else "\n".join(sub_lab[i] for i in present))
        # none / all are single words that stand for the endpoints, so keep them large
        fs = 9.5 if txt in ("none","all") else 8.5 * ({1:1, 2:1, 3:.8, 4:.65, 5:.55}.get(len(present), 1))
        ax.text(x, y, txt, ha="center", va="center", color=tc, fontsize=fs, zorder=4, linespacing=1.02)
    xs = [x for x,_ in pos.values()]; ys = [y for _,y in pos.values()]
    ax.set_xlim(min(xs)-1.6, max(xs)+1.6); ax.set_ylim(min(ys)-1.6, max(ys)+1.6)
    ax.set_aspect("equal"); ax.axis("off")
    ax.set_title(title if title is not None else ag, fontsize=9.5, weight="bold", pad=2)


# ============================================================================
# standalone panel builders, one figure each
# carries  letters and headings in the layout file
# ============================================================================
def save_panel(fig, ab, letter):
    """Write one panel to png, pdf and svg, then check the svg has no live text."""
    stem = f"{fig_dir}/panel_{letter}_{name_to_stem[ab]}"
    fig.savefig(stem + ".png", bbox_inches="tight", dpi=png_dpi)
    fig.savefig(stem + ".pdf", bbox_inches="tight")
    fig.savefig(stem + ".svg", bbox_inches="tight")
    s = open(stem + ".svg").read()
    assert s.count("<text") == 0, "live text present, set svg.fonttype='path'"
    print("saved", stem + ".png / .svg / .pdf")

# Scatter over the key strip
def build_panel_a(ab):
    n2s = meta_maps(ab)[2]; i56, i57 = n2s["S56R"], n2s["T57P"]
    fig = plt.figure(figsize=(5.4, 6.4))
    gs = GridSpec(2, 1, figure=fig, height_ratios=[1, 0.30], hspace=0.30,
                  left=0.16, right=0.97, top=0.98, bottom=0.04)
    ax, ax_key = fig.add_subplot(gs[0]), fig.add_subplot(gs[1])
    m = draw_scatter(ax, ab)
    cats = ["neither","S56R only","T57P only","both"]
    states = [(False,False),(True,False),(False,True),(True,True)]
    counts = [int((m["tok"].apply(lambda t: switch_cat(t,i56,i57)) == c).sum()) for c in cats]
    draw_key(ax_key, [switch_hex[c] for c in cats], ["S56R","T57P"], states, counts,
             markers=reference_status(ab, m))
    save_panel(fig, ab, "A"); return fig

# Boxplots split by S56R, side by side
def build_panel_b(ab):
    state_lab = [("G", "\u2212S56R"), ("M", "+S56R")]
    ylim = (5.9, max(8.3, b_ymax(ab) + 0.12))
    fig, axes = plt.subplots(1, 2, figsize=(6.4, 4.6), sharey=True)
    for ax, (st, lab) in zip(axes, state_lab):
        first = ax is axes[0]
        draw_boxplot_split(ax, ab, st, ylim, ylabel=first, legend=first, xlabel=True)
        ax.set_title(lab, fontsize=10, weight="bold")
    fig.subplots_adjust(left=0.11, right=0.98, top=0.93, bottom=0.13, wspace=0.06)
    save_panel(fig, ab, "B"); return fig

# 2x2 switch-state facets under a background key
def build_panel_c(ab):
    n2s = meta_maps(ab)[2]; i56, i57 = n2s["S56R"], n2s["T57P"]
    m = variant_xy(ab); m["bg"] = m["tok"].apply(lambda t: bg_index(ab, t))
    subs = bg_subsets(ab)
    fig = plt.figure(figsize=(7.6, 9.2))
    gs = GridSpec(3, 2, figure=fig, height_ratios=[0.30, 1, 1], hspace=0.26, wspace=0.10,
                  left=0.12, right=0.97, top=0.98, bottom=0.05)
    ax_key = fig.add_subplot(gs[0, :])
    counts = [int((m["bg"] == i).sum()) for i in range(len(subs))]
    states = [tuple(b in s for b in facet_bg[ab]) for s in subs]
    draw_key(ax_key, bg_hex[:len(subs)], facet_bg[ab], states, counts,
             markers=reference_status(ab, m))
    cells = [(("G","G"),(0,0)), (("M","G"),(0,1)), (("G","M"),(1,0)), (("M","M"),(1,1))]
    for (s56, s57), (r, c) in cells:
        ax = fig.add_subplot(gs[r+1, c])
        draw_facet_scatter(ax, ab, m, s56, s57, xedge=(r == 1), yedge=(c == 0))
    save_panel(fig, ab, "C"); return fig

# Half-violins for both antigens, each over its own dot grid
def build_panel_d(ab):
    ncol = 4 + 2*(2**len(facet_bg[ab]) - 2)
    fig = plt.figure(figsize=(max(11, 0.78*ncol), 8.0))
    gs = GridSpec(2, 1, figure=fig, hspace=0.30, left=0.08, right=0.985, top=0.90, bottom=0.05)
    v_ch = gs[0].subgridspec(2, 1, height_ratios=[0.66,0.34], hspace=0.04)
    ax_vc, ax_dc = fig.add_subplot(v_ch[0]), fig.add_subplot(v_ch[1])
    v_sf = gs[1].subgridspec(2, 1, height_ratios=[0.66,0.34], hspace=0.04)
    ax_vs, ax_ds = fig.add_subplot(v_sf[0]), fig.add_subplot(v_sf[1])
    draw_violin(ax_vc, ax_dc, ab, "CH505TF", legend=True, brackets=True)
    draw_violin(ax_vs, ax_ds, ab, "SF162", legend=True, brackets=True)
    save_panel(fig, ab, "D"); return fig

# Shared uphill/downhill edge-colour key for the landscape panel
def _uphill_downhill_legend(fig, x, y):
    fig.legend(handles=[Line2D([0],[0], color="#333333", lw=2.4, label="uphill (increase in affinity)"),
                        Line2D([0],[0], color="#cfcfcf", lw=2.4, label="downhill (decrease in affinity)")],
               loc="upper center", bbox_to_anchor=(x,y), ncol=2, frameon=False,
               fontsize=8.5, columnspacing=1.4, handlelength=1.6)

# Landscape cubes, colourbar on the left
def build_panel_e(ab):
    labels, regions, n2s = meta_maps(ab)
    subcube = bubble_panel[ab]; sub_idx = [n2s[s] for s in subcube]; sub_lab = [labels[i] for i in sub_idx]
    split_name = bubble_split[ab]
    if split_name is None:
        vmin, vmax = bubble_range(ab, subcube)
        fig = plt.figure(figsize=(12, 7.4))
        gs = GridSpec(1, 2, figure=fig, wspace=0.02, left=0.12, right=0.99, top=0.86, bottom=0.03)
        ax1, ax2 = fig.add_subplot(gs[0]), fig.add_subplot(gs[1])
        draw_bubble(ax1, ab, "CH505TF", sub_idx, sub_lab, vmin, vmax)
        draw_bubble(ax2, ab, "SF162",   sub_idx, sub_lab, vmin, vmax)
        fig.canvas.draw()
        p1, p2 = ax1.get_position(), ax2.get_position()
        cax = fig.add_axes([0.065, p1.y0+p1.height*0.15, 0.012, p1.height*0.68])
        cxc = (p1.x0+p2.x1)/2; box_top = p1.y1
    else:
        vmin, vmax = bubble_range_split(ab, subcube, split_name)
        focal = n2s[split_name]
        split_y_step = 2.9          # tighter vertical spacing between levels
        fig = plt.figure(figsize=(16, 8.4))
        gs = GridSpec(2, 2, figure=fig, wspace=0.0, hspace=0.05, left=0.13, right=0.99, top=0.88, bottom=0.03)
        axes = {}
        colstate = [("G", f"\u2212{split_name}"), ("M", f"+{split_name}")]
        for ri, ag in enumerate(antigens):
            for ci, (st, clab) in enumerate(colstate):
                ax = fig.add_subplot(gs[ri,ci]); axes[(ri,ci)] = ax
                fn = avg_val_fn(ab, ag, sub_idx, focal, st)
                draw_bubble(ax, ab, ag, sub_idx, sub_lab, vmin, vmax, val_fn=fn,
                            title=(clab if ri==0 else ""), y_step=split_y_step)
        fig.canvas.draw()
        for ri, ag in enumerate(antigens):
            p = axes[(ri,0)].get_position()
            fig.text(p.x0-0.010, (p.y0+p.y1)/2, ag, rotation=90, ha="right", va="center",
                     fontsize=11, fontweight="bold")
        p_bl, p_tr = axes[(1,0)].get_position(), axes[(0,1)].get_position()
        block_y0, block_y1 = p_bl.y0, p_tr.y1
        cb_h = (block_y1 - block_y0) * 0.74
        cb_y0 = block_y0 + (block_y1 - block_y0 - cb_h) / 2          # centred on the cube block
        cax = fig.add_axes([0.045, cb_y0, 0.011, cb_h])
        cxc = (axes[(0,0)].get_position().x0 + p_tr.x1)/2; box_top = p_tr.y1
    sm = ScalarMappable(Normalize(vmin,vmax), plt.get_cmap(bubble_cmap)); sm.set_array([])
    cb = fig.colorbar(sm, cax=cax); cb.set_label(lab_inf, fontsize=8)
    cax.yaxis.set_ticks_position("right"); cax.yaxis.set_label_position("left")
    h_in = fig.get_size_inches()[1]
    _uphill_downhill_legend(fig, cxc, box_top + 0.38/h_in)
    save_panel(fig, ab, "E"); return fig

## Generate panels: 1NC9

In [ ]:
for fn in (build_panel_a, build_panel_b, build_panel_c, build_panel_d, build_panel_e):
    fn("1NC9")

## Generate panels: CH235.12

In [ ]:
for fn in (build_panel_a, build_panel_b, build_panel_c, build_panel_d, build_panel_e):
    fn("CH235.12")